# Part 5e: Generate the bitfile

This notebook runs the full flow of the `VitisUnified` backend: HLS synthesis, linking the kernel into the SoC platform with `v++`, place and route, and bitstream generation. The result is written to `export/` in the project directory:

| File | Purpose |
|------|---------|
| `system.bit` | the bitstream |
| `system.hwh` | the hardware handoff file that PYNQ uses to discover the IPs and their addresses |
| `*_driver.py` | the PYNQ driver for the selected `axi_mode` |

Copy these three files to the board and load them with the driver from a PYNQ notebook.

> **This step takes a long time** (typically well over an hour).
>
> With `BOARD = 'zcu102'` and `AXI_MODE = 'axi_stream'`, or with `BOARD = 'kv260'`, the backend first builds the platform from a Tcl script, so **Vivado must be on `PATH` as well**. This notebook adds `$XILINX_VIVADO/bin` for that reason.

## Settings

Set the target board here. The backend picks the platform (`.xpfm` or `.xsa`) from its `supported_boards.json` using the board and the AXI mode, so nothing else needs to be configured. To add another board, see `5_vu_f_platform_setup.md`.

| Variable | Meaning |
|----------|---------|
| `BOARD` | `'zcu102'` or `'kv260'` |
| `AXI_MODE` | `'axi_master'` (default) or `'axi_stream'`; see the *AXI interface modes* page of the hls4ml docs |

The Vitis tools are located through the `XILINX_VITIS` environment variable, as in the other parts of this tutorial.

In [ ]:
import os

os.environ['KERAS_BACKEND'] = 'tensorflow'

import numpy as np
import sys

sys.path.append('../..')
from models import create_simple_unet
import hls4ml

os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']
os.environ['PATH'] = os.environ['XILINX_VIVADO'] + '/bin:' + os.environ['PATH']

BOARD = 'zcu102'  # or 'kv260'
AXI_MODE = 'axi_master'  # or 'axi_stream'

DATA_DIR = '../../data/vitis_unified'
os.makedirs(DATA_DIR, exist_ok=True)
np.random.seed(0)

## Create the model and some test data

We use a tiny U-Net with a single skip connection, defined in `models.py`. The weights are left untrained on purpose: the goal of this part is to check that every stage of the accelerator flow reproduces the same numbers, not to reach a good accuracy. A `Concatenate` skip connection is a good stress test for the streaming interface, because it forces two branches to be merged again.

In [ ]:
X = np.random.rand(10000, 4, 4, 1).astype(np.float32)
np.save(f'{DATA_DIR}/X_part5e.npy', X)
print(f'Input shape: {X.shape}')

model = create_simple_unet()
model.summary()

## Convert and predict

We save the bridge predictions for 10000 inputs alongside the inputs, so that they can be copied to the board together with the bitfile and used to validate the hardware.

The `VitisUnified` backend takes the same arguments as the `Vitis` backend, plus:

| Argument | Meaning |
|----------|---------|
| `board` | the target board; the FPGA part and the platform are looked up from it |
| `axi_mode` | `'axi_master'` or `'axi_stream'`, the interface between the PS and the kernel |
| `input_type`, `output_type` | the data type on the AXI interface (`'float'` here, so the driver can send `float32` NumPy arrays directly) |

In [ ]:
config = hls4ml.utils.config_from_keras_model(model, granularity='name', backend='Vitis')

output_dir = os.path.abspath(f'../../hls4ml_prjs/hls4ml_prj_part5e_{AXI_MODE}')

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config,
    backend='VitisUnified',
    io_type='io_stream',
    output_dir=output_dir,
    board=BOARD,
    axi_mode=AXI_MODE,
    input_type='float',
    output_type='float',
    clock_period=10,
)
hls_model.compile()

y_ref = hls_model.predict(X)
np.save(f'{DATA_DIR}/y_part5e.npy', y_ref)
print(f'Predictions saved: shape {np.array(y_ref).shape}')

## Synthesize and generate the bitfile

`bitfile=True` runs everything: HLS synthesis, platform generation (if needed), `v++ --link`, and Vivado implementation. **This cell takes a long time.**

In [ ]:
hls_model.build(synth=True, bitfile=True, log_to_stdout=True)

In [ ]:
export_dir = f'{output_dir}/export'
print('Files to copy to the board:')
for f in sorted(os.listdir(export_dir)):
    print(f'  {export_dir}/{f}')